In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import seaborn as sns
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
df = pd.read_excel("Final_1.xlsx")
df = df.drop(columns=['Unnamed: 0', 'Unnamed: 0.1', 'Unnamed: 0.2'])
label_encoder = LabelEncoder()
df['Well Name'] = label_encoder.fit_transform(df['Well Name'])
df['Formation'] = label_encoder.fit_transform(df['Formation'])
df_train = df[df['Well Name'] != 1]
df_blind = df[df['Well Name'] == 1]
df

In [ ]:
df_blind

In [ ]:
facies_colors = ['#F4D03F', '#F5B041','#DC7633','#6E2C00',
       '#1B4F72','#2E86C1', '#AED6F1', '#A569BD', '#196F3D']

facies_labels = ['SS', 'CSiS', 'FSiS', 'SiSh', 'MS',
                 'WS', 'D','PS', 'BS']

# facies_labels = ['facies_{}'.format(i) for i in range(1, len(df['Facies'].unique()) + 1)]

facies_color_map = {label: color for label, color in zip(facies_labels, facies_colors)}

df_train['FaciesLabels'] = df_train['Facies'].apply(lambda x: facies_labels[x - 1])


In [ ]:
correct_facies_labels = df_train['Facies'].values
feature_vectors = df_train[['GR', 'ILD_log10', 'PHIND', 'RELPOS', 'Formation', 'Well Name', 'Core_Porosity', 'Core_Perm']]

In [ ]:
target_facies_labels = df_blind['Facies'].values
blind_feature = df_blind[['GR', 'ILD_log10', 'PHIND', 'RELPOS', 'Formation', 'Well Name', 'Core_Porosity', 'Core_Perm']]

In [ ]:
blind_feature

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score

# Define parameter grid
param_grid = {
    'alpha': [0.0001, 0.001, 0.01],
    'max_iter': [500, 1000, 1500],
    'learning_rate_init': [0.0001, 0.001, 0.01],
    'hidden_layer_sizes': [(100,), (200,), (300,)],
}

NNC_model = MLPClassifier(solver='adam', batch_size=10)

scorer = make_scorer(f1_score, average='weighted')


grid_search = GridSearchCV(estimator=NNC_model, param_grid=param_grid, scoring=scorer, cv=5)

grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print("Best Parameters:", best_params)

yhat_best = best_model.predict(X_test)

f1_best = f1_score(y_test, yhat_best, average='weighted')
print('Best Model F1 Score:', f1_best)



In [ ]:
from sklearn import preprocessing
from sklearn.model_selection import train_test_split

scaler = preprocessing.StandardScaler().fit(feature_vectors)
scaled_features = scaler.transform(feature_vectors)

X_train, X_test, y_train, y_test = train_test_split(
        scaled_features, correct_facies_labels, test_size=0.2, random_state=10)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
RFC_model = RandomForestClassifier(max_depth=12, n_estimators=20, max_features=6).fit(X_train, y_train)
yhat_RFC = RFC_model.predict(X_test)


from sklearn.neural_network import MLPClassifier
NNC_model = MLPClassifier(alpha=0.001, max_iter=1000, learning_rate_init=0.001, 
                          solver='adam', batch_size=10, hidden_layer_sizes=200).fit(X_train, y_train)
yhat_NNC = NNC_model.predict(X_test)


from sklearn.tree import DecisionTreeClassifier
DT_model = DecisionTreeClassifier(criterion="entropy", max_depth=25, min_samples_split=3)
DT_model.fit(X_train, y_train)
yhat_DT = DT_model.predict(X_test)



from sklearn.metrics import f1_score
f1RFC = f1_score(y_test, yhat_RFC, average='weighted')
f1NNC =  f1_score(y_test, yhat_NNC, average='weighted')
f1DT = f1_score(y_test, yhat_DT, average='weighted')

print('RFC:', f1RFC)
print('NNC:', f1NNC)
print('DTC:', f1DT)


In [ ]:
# from sklearn.ensemble import RandomForestClassifier
# RFC_model = RandomForestClassifier(max_depth=12, n_estimators=20, max_features=6).fit(X_train, y_train)
# yhat_RFC = RFC_model.predict(blind_feature)


# from sklearn.neural_network import MLPClassifier
# NNC_model = MLPClassifier(alpha=0.001, max_iter=1000, learning_rate_init=0.001, 
#                           solver='adam', batch_size=10, hidden_layer_sizes=200).fit(X_train, y_train)
yhat_NNC = NNC_model.predict(blind_feature)


# from sklearn.tree import DecisionTreeClassifier
# DT_model = DecisionTreeClassifier(criterion="entropy", max_depth=25, min_samples_split=3)
# DT_model.fit(X_train, y_train)
# yhat_DT = DT_model.predict(blind_feature)







# # from sklearn.metrics import f1_score/
# f1RFC = f1_score(target_facies_labels, yhat_RFC, average='weighted')
f1NNC =  f1_score(target_facies_labels, yhat_NNC, average='weighted')
# f1DT = f1_score(target_facies_labels, yhat_DT, average='weighted')

# print('RFC:', f1RFC)
print('NNC:', f1NNC)
# print('DTC:', f1DT)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
RFC_model = RandomForestClassifier(max_depth=12, n_estimators=20, max_features=6).fit(X_train, y_train)
yhat_RFC = RFC_model.predict(X_test)


from sklearn.neural_network import MLPClassifier
NNC_model = MLPClassifier(alpha=0.001, max_iter=1000, learning_rate_init=0.001, 
                          solver='adam', batch_size=10, hidden_layer_sizes=200).fit(X_train, y_train)
yhat_NNC = NNC_model.predict(X_test)


from sklearn.tree import DecisionTreeClassifier
DT_model = DecisionTreeClassifier(criterion="entropy", max_depth=25, min_samples_split=3)
DT_model.fit(X_train, y_train)
yhat_DT = DT_model.predict(X_test)



from sklearn.metrics import f1_score
f1RFC = f1_score(df_blind_t, yhat_RFC, average='weighted')
f1NNC =  f1_score(df_blind_t, yhat_NNC, average='weighted')
f1DT = f1_score(df_blind_t, yhat_DT, average='weighted')

print('RFC:', f1RFC)
print('NNC:', f1NNC)
print('DTC:', f1DT)


In [ ]:
# Define a function to display confusion matrix with metrics
def display_cm(conf_matrix, labels, hide_zeros=False, display_metrics=False):
    """Display confusion matrix with optional metrics."""
    # Calculate metrics
    if display_metrics:
        precision = np.diag(conf_matrix) / np.sum(conf_matrix, axis=0)
        recall = np.diag(conf_matrix) / np.sum(conf_matrix, axis=1)
        f1_score = 2 * (precision * recall) / (precision + recall)
    
    # Print confusion matrix
    print("Confusion Matrix:")
    for i, label in enumerate(labels):
        if not hide_zeros or np.sum(conf_matrix[i]) > 0:
            print(f"{label}: {conf_matrix[i]}")
    
    # # Print metrics if requested
    # if display_metrics:
    #     print("\nMetrics:")
    #     for i, label in enumerate(labels):
    #         print(f"{label}: Precision={precision[i]:.2f}, Recall={recall[i]:.2f}, F1-Score={f1_score[i]:.2f}")


In [ ]:
from sklearn.metrics import jaccard_score, f1_score, confusion_matrix

jRFC = jaccard_score(y_test, yhat_RFC, average='weighted')
f1RFC = f1_score(y_test, yhat_RFC, average='weighted')

# print("RFC Jaccard index: %.3f" % jRFC)
print("RFC F1-score: %.3f" % f1RFC)

conf_RFC = confusion_matrix(y_test, yhat_RFC)
display_cm(conf_RFC, facies_labels, hide_zeros=True, display_metrics=True)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Function to plot confusion matrix
def plot_confusion_matrix(cm, classes):
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Confusion Matrix')
    plt.colorbar()
    tick_marks = range(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()


conf_RFC = confusion_matrix(y_test, yhat_RFC)

# Plot confusion matrix for Random Forest Classifier
plot_confusion_matrix(conf_RFC, facies_labels)

In [ ]:
jNNC = jaccard_score(y_test, yhat_NNC, average='weighted')
f1NNC =  f1_score(y_test, yhat_NNC, average='weighted')

print("NNC Jaccard index: %.3f" %jNNC)
print("NNC F1-score: %.3f" %f1NNC )

conf_NNC = confusion_matrix(y_test, yhat_NNC)
display_cm(conf_NNC, facies_labels, hide_zeros=True, display_metrics=True)

In [ ]:
conf_NNC = confusion_matrix(y_test, yhat_NNC)
plot_confusion_matrix(conf_NNC, facies_labels)

In [ ]:
jDT = jaccard_score(y_test, yhat_NNC, average='weighted')
f1DT = f1_score(y_test, yhat_DT, average='weighted')
print("DT Jaccard index: %.3f" % jDT )
print("DT F1-score: %.3f" % f1DT )

conf_DT = confusion_matrix(y_test, yhat_DT)
display_cm(conf_DT, facies_labels, hide_zeros=True, display_metrics=True)

In [ ]:
conf_DT = confusion_matrix(y_test, yhat_DT)

plot_confusion_matrix(conf_DT, facies_labels)

In [ ]:
y_blind = blind['Facies'].values
blind_features = blind.drop(['Formation', 'Well Name', 'Depth','Facies','FaciesLabels'], axis=1)
X_blind = scaler.transform(blind_features)

In [ ]:
blind.to_csv('blind.csv')

In [ ]:
blind_features1 = blind[['GR', 'ILD_log10', 'DeltaPHI', 'PHIND', 'PE', 'NM_M','RELPOS']]

blind_features1

#### Predicting facies for blind well by all models

In [ ]:
#RFC
yhat_blind_RFC = RFC_model.predict(X_blind)
blind['RFC_Pred'] = yhat_blind_RFC

#NNC
yhat_blind_NNC = NNC_model.predict(X_blind)
blind['NNC_Pred'] = yhat_blind_NNC

#DT
yhat_blind_DT = DT_model.predict(X_blind)
blind['DT_Pred'] = yhat_blind_DT


In [ ]:
def compare_all_facies(logs, Pred, facies_colors):
    logs = logs.sort_values(by='Depth')
    cmap_facies = colors.ListedColormap(facies_colors[0:len(facies_colors)], 'indexed')
    ztop=logs.Depth.min(); zbot=logs.Depth.max()
    
    cluster_initial = np.repeat(np.expand_dims(logs['Facies'].values,1), 100, 1)
    cluster = np.repeat(np.expand_dims(logs[Pred].values,1), 100, 1)

   
    f, ax = plt.subplots(nrows=1, ncols=2, figsize=(3, 6))

    im1 = ax[0].imshow(cluster_initial, interpolation='none', aspect='auto',
                       cmap=cmap_facies,vmin=1,vmax=9)

    im8 = ax[1].imshow(cluster, interpolation='none', aspect='auto',
                       cmap=cmap_facies,vmin=1,vmax=9)
       
    
    divider = make_axes_locatable(ax[1])
    cax = divider.append_axes("right", size="10%", pad=0.05)
    cbar=plt.colorbar(im8, cax=cax)
    cbar.set_label((5*' ').join([' SS ', 'CSiS', 'FSiS', 
                                'SiSh', ' MS ', ' WS ', ' D  ', 
                                ' PS ', ' BS ']))
    cbar.set_ticks(range(0,1)); cbar.set_ticklabels('')
    
    for i in range(len(ax)-8):
        ax[i].set_ylim(ztop,zbot)
        ax[i].invert_yaxis()
        ax[i].grid()
        ax[i].locator_params(axis='x', nbins=2)
    
    ax[0].set_xlabel('Facies'); ax[1].set_xlabel(Pred)

    
    ax[0].set_yticklabels([]) ; ax[1].set_yticklabels([]) 
    
    ax[0].set_xticklabels([]); ax[1].set_xticklabels([]) 


    f.suptitle('Prediction in well: %s'%logs.iloc[0]['Well Name'], fontsize=14,y=0.94)

In [ ]:
compare_all_facies(blind,'NNC_Pred', facies_colors)